In [ ]:
import torch

print("=" * 50)
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
else:
    print("No GPU detected!")
print("=" * 50)

In [ ]:
Output
==================================================
PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4
CUDA Version: 12.8
==================================================


In [ ]:
%%capture

# Install Unsloth
!pip install unsloth

# Install remaining dependencies
!pip install \
    transformers \
    datasets \
    trl \
    peft \
    accelerate \
    bitsandbytes \
    sentencepiece

In [ ]:
import torch
import transformers
import datasets
import peft
import trl
import unsloth

print("All libraries imported successfully!")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

print("Current Working Directory:", os.getcwd())
print("\nFiles in /content:")

for file in os.listdir("/content"):
    print(file)

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset(
    "json",
    data_files="train_data.jsonl",
    split="train"
)

test_dataset = load_dataset(
    "json",
    data_files="test_data.jsonl",
    split="train"
)

In [ ]:
print("=" * 50)
print("Training Examples :", len(train_dataset))
print("Test Examples     :", len(test_dataset))
print("=" * 50)

In [ ]:
messages = sample["messages"]

for message in messages:
    print("=" * 60)
    print("ROLE:", message["role"].upper())
    print("=" * 60)
    print(message["content"])
    print()

============================================================
ROLE: SYSTEM
============================================================
You are an Edge Logging Firewall. Analyze backend production logs, ignore irrelevant noise, redact sensitive PII, and output ONLY valid JSON matching the telemetry schema.

============================================================
ROLE: USER
============================================================
2026-06-26 07:35:07,144 | INFO     | email-service | [req_716717cf] - Authentication check successful for session user=usr_7308 scopes=['read', 'write']
2026-06-26 07:35:07,436 | INFO     | email-service | [req_716717cf] - Pub/Sub message acknowledged from subscription projects/gcp-prod/topics/billing-sync-sub msg_id=04ac745bce10
2026-06-26 07:35:08,136 | INFO     | email-service | [req_716717cf] - Handled HTTP DELETE /api/v1/payments/charge status=201 duration=133.24ms
2026-06-26 07:35:08,729 | INFO     | email-service | [req_716717cf] - Request payload preview: user_email=carrollchristopher@example.net client_ip=147.15.178.175 status=processing
2026-06-26 07:35:09,523 | INFO     | email-service | [req_716717cf] - Request payload preview: user_email=connor43@example.net client_ip=96.99.178.130 status=processing
2026-06-26 07:35:10,205 | ERROR    | email-service | [req_716717cf] - [Pub/Sub failure] GoogleCloudError: Pub/Sub publish request failed on topic 'projects/gcp-prod/topics/order-events': DeadlineExceeded
2026-06-26 07:35:10,844 | WARNING  | email-service | [req_716717cf] - Retrying operation after failure (Pub/Sub failure) attempt 1 of 3 - backoff 1.5s...

============================================================
ROLE: ASSISTANT
============================================================
{"error_type": "PubSubFailure", "affected_service": "email-service", "root_cause": "Publish request deadline exceeded", "pii_redacted": true, "summary": "Email service failed to synchronize payloads via Pub/Sub when the network deadline was exceeded."}



In [ ]:
import json

assistant_output = sample["messages"][2]["content"]

parsed = json.loads(assistant_output)

print("Assistant JSON is valid!\n")

print(json.dumps(parsed, indent=4))

In [ ]:
from unsloth import FastLanguageModel

# 1. Define Model Parameters
max_seq_length = 2048 # 2048 tokens is plenty for your log clusters
dtype = None # Auto-detects based on your T4 GPU
load_in_4bit = True # This shrinks the 1B model down to ~700MB in GPU memory!

print("Downloading and loading the base model in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Attach the LoRA Adapters (The "Blank Brain")
print("Attaching QLoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The rank of the adapter. 16 is the industry standard sweet spot.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Set to 0 for highly optimized Unsloth training
    bias = "none",    
    use_gradient_checkpointing = "unsloth", # Saves massive amounts of VRAM
    random_state = 3407,
)

print("Model and Adapters loaded successfully! You are ready to format data.")

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# 1. Tell the tokenizer we are using the ChatML format you built
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

# 2. A simple function to apply the template to your JSON rows
def formatting_prompts_func(examples):
    convos = examples["messages"]
    # This turns your JSON array into the strict <|im_start|> string format
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts, }

# 3. Load your files
print("Loading train and test datasets...")
train_dataset = load_dataset("json", data_files="train_data.jsonl", split="train")
test_dataset = load_dataset("json", data_files="test_data.jsonl", split="train")

# 4. Apply the mapping
print("Applying ChatML formatting...")
train_dataset = train_dataset.map(formatting_prompts_func, batched = True,)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True,)

print(f"Success! Prepared {len(train_dataset)} training rows and {len(test_dataset)} testing rows.")

 WARNING:unsloth.chat_templates:Unsloth: Will map <|im_end|> to EOS = <|eot_id|>.
Loading train and test datasets...
Applying ChatML formatting...
Map: 100% 400/400 [00:00<00:00, 7699.11 examples/s]Map: 100% 100/100 [00:00<00:00, 3488.68 examples/s]Success! Prepared 400 training rows and 100 testing rows.


In [ ]:
from tqdm import tqdm

max_tokens = 0
min_tokens = float("inf")
total_tokens = 0
longest_index = -1
shortest_index = -1

for i in tqdm(range(len(train_dataset))):
    token_count = len(tokenizer(train_dataset[i]["text"])["input_ids"])

    total_tokens += token_count

    if token_count > max_tokens:
        max_tokens = token_count
        longest_index = i

    if token_count < min_tokens:
        min_tokens = token_count
        shortest_index = i

print("=" * 60)
print(f"Training Examples : {len(train_dataset)}")
print(f"Shortest Example  : {min_tokens} tokens (Index {shortest_index})")
print(f"Longest Example   : {max_tokens} tokens (Index {longest_index})")
print(f"Average Length    : {total_tokens / len(train_dataset):.2f} tokens")
print("=" * 60)

100%|██████████| 400/400 [00:00<00:00, 610.25it/s]============================================================
Training Examples : 400
Shortest Example  : 234 tokens (Index 211)
Longest Example   : 1316 tokens (Index 206)
Average Length    : 607.12 tokens
============================================================



In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Configuring the SFT Trainer...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Keep False for ChatML conversations

    args=TrainingArguments(
        # =========================
        # Training Configuration
        # =========================
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_steps=5,

        # =========================
        # Mixed Precision
        # =========================
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),

        # =========================
        # Evaluation
        # =========================
        eval_strategy="epoch",

        # =========================
        # Disable Checkpoint Saving
        # (Workaround for TRL PicklingError)
        # =========================
        save_strategy="no",

        # =========================
        # Optimizer
        # =========================
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",

        # =========================
        # Logging
        # =========================
        logging_steps=10,
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Starting training...\n")

trainer_stats = trainer.train()

print("\nTraining completed successfully!")

 Configuring the SFT Trainer...
Starting training...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)

    
      
      
      [100/100 05:38, Epoch 2/2]
    
    
  
 
      Epoch
      Training Loss
      Validation Loss
    
  
  
    
      1
      0.627570
      0.605858
    
    
      2
      0.542408
      0.569115
    
  
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)

Training completed successfully!


In [ ]:
from unsloth import FastLanguageModel
import torch

# Switch model to inference mode
FastLanguageModel.for_inference(model)

# A completely new log (NOT from your training dataset)
test_log = """
<LOG_START>
2026-08-10T14:20:13.512Z [INFO] [payment-service] Starting checkout request

{"timestamp":"2026-08-10T14:20:14.021Z",
 "severity":"INFO",
 "serviceContext":{"service":"payment-service","version":"payment-service-0123"},
 "trace":"projects/prod/traces/abcd123456789",
 "message":"Authenticated user user_9182"}

{"timestamp":"2026-08-10T14:20:14.638Z",
 "severity":"ERROR",
 "serviceContext":{"service":"payment-service","version":"payment-service-0123"},
 "trace":"projects/prod/traces/abcd123456789",
 "message":"Exception during POST /api/v1/payments/checkout: KeyError: 'payment_token'"}

Traceback (most recent call last):
  File "/app/api/payment.py", line 208, in process_payment
    token = request["payment_token"]
KeyError: 'payment_token'

<LOG_END>
"""

messages = [
    {
        "role": "system",
        "content": "You are an Edge Logging Firewall. Read the raw logs, ignore noise, redact PII, and output ONLY the strict JSON telemetry schema."
    },
    {
        "role": "user",
        "content": test_log
    }
]

# Apply ChatML template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

# Generate prediction
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=150,
    temperature=0.0,
    do_sample=False,
)

response = tokenizer.decode(
    outputs[0][inputs.shape[1]:],
    skip_special_tokens=True,
)

print("=" * 60)
print("MODEL OUTPUT")
print("=" * 60)
print(response)

In [ ]:
import os
from google.colab import files

print("=" * 60)
print("Exporting model to GGUF...")
print("=" * 60)

model.save_pretrained_gguf(
    "edge_firewall_model",
    tokenizer,
    quantization_method="q4_k_m",
)

print("\nSearching for generated GGUF file...\n")

gguf_files = [
    f for f in os.listdir("edge_firewall_model")
    if f.endswith(".gguf")
]

if len(gguf_files) == 0:
    raise FileNotFoundError("No GGUF file was generated!")

gguf_path = os.path.join(
    "edge_firewall_model",
    gguf_files[0]
)

print(f"Found: {gguf_files[0]}")
print("\nStarting download...\n")

files.download(gguf_path)

 ============================================================
Exporting model to GGUF...
============================================================
Unsloth: Merging model weights to 16-bit format...
config.json: 100% 894/894 [00:00<00:00, 40.8kB/s]Unsloth: Restored added_tokens_decoder metadata in edge_firewall_model/tokenizer_config.json.
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]model.safetensors: 100% 2.47G/2.47G [00:33<00:00, 83.7MB/s]Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:33<00:00, 33.42s/it]
Note: tokenizer.model not found (this is OK for non-SentencePiece models)
Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:44<00:00, 44.94s/it]
Unsloth: Merge process complete. Saved to `/content/edge_firewall_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9950-mix-53618c5 (app-b9950-mix-53618c5-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['edge_firewall_model_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['edge_firewall_model_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model edge_firewall_model_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to edge_firewall_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f edge_firewall_model_gguf/Modelfile

Searching for generated GGUF file...

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
/tmp/ipykernel_2609/4196048336.py in <cell line: 0>()
     20 
     21 if len(gguf_files) == 0:
---> 22     raise FileNotFoundError("No GGUF file was generated!")
     23 
     24 gguf_path = os.path.join(

FileNotFoundError: No GGUF file was generated!

In [ ]:
import os
from google.colab import files

# 1. Define the output file name
gguf_file_name = "edge-firewall-llama-3.2-1B-q4_k_m.gguf"

print("Merging adapters and converting to 4-bit GGUF...")
print("This will take about 5-10 minutes. Do not close the tab!")

# 2. Convert and Save
# We use "q4_k_m" (4-bit quantization). It is the industry sweet spot 
# for keeping the model smart while making the file size incredibly small (~700MB).
model.save_pretrained_gguf(
    "edge_firewall_model", 
    tokenizer, 
    quantization_method = "q4_k_m"
)

# 3. Locate the generated file (Unsloth saves it inside the directory)
generated_gguf_path = f"edge_firewall_model/{gguf_file_name}"

if os.path.exists(generated_gguf_path):
    print(f"Success! Model saved at {generated_gguf_path}")
    print("Initiating download to your local machine...")
    files.download(generated_gguf_path)
else:
    # Fallback in case Unsloth names the output file slightly differently based on the base model name
    print("Finding the .gguf file...")
    for file in os.listdir("edge_firewall_model"):
        if file.endswith(".gguf"):
            print(f"Downloading {file}...")
            files.download(f"edge_firewall_model/{file}")
            break

Merging adapters and converting to 4-bit GGUF...
This will take about 5-10 minutes. Do not close the tab!
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 849.57it/s]
Note: tokenizer.model not found (this is OK for non-SentencePiece models)
Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:32<00:00, 32.79s/it]
Unsloth: Merge process complete. Saved to `/content/edge_firewall_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['edge_firewall_model_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['edge_firewall_model_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model edge_firewall_model_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to edge_firewall_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f edge_firewall_model_gguf/Modelfile
Finding the .gguf file...
